<a href="https://colab.research.google.com/github/sathwik10126/NLP/blob/Main/2403A52245_NLP_Assignment_12_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [192]:
# Install Gensim for Word2Vec
!pip install gensim

In [193]:
# Numerical operations
import numpy as np

# Keras modules
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D
from tensorflow.keras.layers import GlobalMaxPooling1D, Dense, Concatenate
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Gensim for Word2Vec
from gensim.models import Word2Vec

In [194]:
texts = [
"amazing storyline",
"i enjoyed this movie",
"boring and slow film",
"fantastic performance",
"waste of time",
"this film is excellent",
"great direction",
"i really liked this film",
"bad acting",
"superb movie experience",
"not worth watching"
]

# Labels: 1 = positive, 0 = negative
labels = [1,1,0,1,0,1,1,1,0,1,0]

In [195]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
word_index = tokenizer.word_index
print(f'Vocabulary size: {len(word_index)}')
print(f'Number of sequences: {len(sequences)}')

Vocabulary size: 28
Number of sequences: 11


In [196]:
print(sequences)

[[5, 6], [3, 7, 1, 4], [8, 9, 10, 2], [11, 12], [13, 14, 15], [1, 2, 16, 17], [18, 19], [3, 20, 21, 1, 2], [22, 23], [24, 4, 25], [26, 27, 28]]


In [197]:
max_length = 10
X = pad_sequences(sequences, maxlen=max_length)
y = np.array(labels)
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

X shape: (11, 10)
y shape: (11,)


In [198]:
X

array([[ 0,  0,  0,  0,  0,  0,  0,  0,  5,  6],
       [ 0,  0,  0,  0,  0,  0,  3,  7,  1,  4],
       [ 0,  0,  0,  0,  0,  0,  8,  9, 10,  2],
       [ 0,  0,  0,  0,  0,  0,  0,  0, 11, 12],
       [ 0,  0,  0,  0,  0,  0,  0, 13, 14, 15],
       [ 0,  0,  0,  0,  0,  0,  1,  2, 16, 17],
       [ 0,  0,  0,  0,  0,  0,  0,  0, 18, 19],
       [ 0,  0,  0,  0,  0,  3, 20, 21,  1,  2],
       [ 0,  0,  0,  0,  0,  0,  0,  0, 22, 23],
       [ 0,  0,  0,  0,  0,  0,  0, 24,  4, 25],
       [ 0,  0,  0,  0,  0,  0,  0, 26, 27, 28]], dtype=int32)

In [199]:
sentences = [text.split() for text in texts]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences,
    vector_size=50,
    window=3,
    min_count=1
)

In [200]:
vocab_size = len(word_index) + 1
embedding_dim = 50
embedding_matrix = np.zeros((vocab_size, embedding_dim))

sentences = [text.split() for text in texts]
w2v_model = Word2Vec(sentences, vector_size=50, window=3, min_count=1)

for word, index in word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[index] = w2v_model.wv[word]

In [201]:
embedding_matrix[1]

array([-0.01631586,  0.00898562, -0.00827116,  0.00164768,  0.01699369,
       -0.00892899,  0.00903548, -0.0135738 , -0.0070963 ,  0.01878764,
       -0.00315715,  0.00063707, -0.00827949, -0.01536571, -0.00302365,
        0.00494396, -0.00177517,  0.01106993, -0.00549091,  0.00451953,
        0.01090931,  0.01669789, -0.00290373, -0.01841136,  0.00874   ,
        0.00114678,  0.01487957, -0.00162147, -0.00528142, -0.01750105,
       -0.0017141 ,  0.00565104,  0.01080109,  0.01410451, -0.01140432,
        0.00371285,  0.01217692, -0.00960362, -0.0062134 ,  0.01359912,
        0.00326371,  0.00038122,  0.0069451 ,  0.00043395,  0.01923963,
        0.0101201 , -0.01782971, -0.01408546,  0.00180485,  0.01278598])

In [202]:
input_layer = Input(shape=(max_length,))

In [203]:
# Re-create embedding layer with updated vocab_size
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    trainable=False
)(input_layer)

In [204]:
conv1 = Conv1D(filters=100, kernel_size=3, activation='relu')(embedding_layer)

conv2 = Conv1D(filters=100, kernel_size=4, activation='relu')(embedding_layer)

conv3 = Conv1D(filters=100, kernel_size=5, activation='relu')(embedding_layer)

In [205]:
pool1 = GlobalMaxPooling1D()(conv1)
pool2 = GlobalMaxPooling1D()(conv2)
pool3 = GlobalMaxPooling1D()(conv3)

In [206]:
merged = Concatenate()([pool1, pool2, pool3])

In [207]:
dense = Dense(10, activation='relu')(merged)

In [208]:
output = Dense(1, activation='sigmoid')(dense)

In [209]:
# Re-create embedding layer and model with updated dimensions
input_layer = Input(shape=(max_length,))
embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], trainable=False)(input_layer)

conv1 = Conv1D(filters=100, kernel_size=3, activation='relu')(embedding_layer)
conv2 = Conv1D(filters=100, kernel_size=4, activation='relu')(embedding_layer)
conv3 = Conv1D(filters=100, kernel_size=5, activation='relu')(embedding_layer)

pool1 = GlobalMaxPooling1D()(conv1)
pool2 = GlobalMaxPooling1D()(conv2)
pool3 = GlobalMaxPooling1D()(conv3)

merged = Concatenate()([pool1, pool2, pool3])
dense = Dense(10, activation='relu')(merged)
output = Dense(1, activation='sigmoid')(dense)

model = Model(inputs=input_layer, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [210]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [211]:
model.summary()

Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_13      │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_14        │ (None, 10, 50)    │      1,450 │ input_layer_13[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_45 (Conv1D)  │ (None, 8, 100)    │     15,100 │ embedding_14[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_46 (Conv1D)  │ (None, 7, 100)    │     20,100 │ embedding_14[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_47 (Conv1D)  │ (None, 6, 100)    │     25,100 │ embedding_14[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_45[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_46[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_47[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_15      │ (None, 300)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_30 (Dense)    │ (None, 10)        │      3,010 │ concatenate_15[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_31 (Dense)    │ (None, 1)         │         11 │ dense_30[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 64,771 (253.01 KB)

 Trainable params: 63,321 (247.35 KB)

 Non-trainable params: 1,450 (5.66 KB)

In [212]:
model.fit(X, y, epochs=10, batch_size=2)

Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.6364 - loss: 0.6915
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6364 - loss: 0.6800 
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6364 - loss: 0.6648
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6364 - loss: 0.6534 
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6364 - loss: 0.6393 
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6364 - loss: 0.6244 
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6364 - loss: 0.6079
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6364 - loss: 0.5907
Epoch 9/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6364 - loss: 0.5739
Epoch 10/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6364 - loss: 0.5561    


In [213]:
test_text = ["excellent acting"]
seq = tokenizer.texts_to_sequences(test_text)
pad = pad_sequences(seq, maxlen=max_length)
prediction = model.predict(pad)
print(f'Prediction: {prediction}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
Prediction: [[0.5994241]]
